In [ ]:
# you modify the following in the command line interface, like

### notebook config ###

input_path = None # where your metadata is (necessary)
output_path = None # where to store your results
model_path = None # where to store your model
batch_size = 64
num_workers = 0 # speeds up audio processing
base_folder = '/nfs/turbo/umor-sethtem/log-spectrogram-128mel-2048fft-512hop-5dur' # where high-level acoustics data files lie
mel_time_size = 313 # target time dimension size for mel spectrograms (applies random padding if specified)
n_samples = -1


In [ ]:
# paths to folders and files
collection_map = {
    'iNat': 'birdclef-2026/train_audio',
    'XC': 'birdclef-2026/train_audio',
    'iNat2': 'birdclef-2025/train_audio',
    'XC2': 'birdclef-2025/train_audio',
    'CSA': 'birdclef-2025/train_audio',
    'esc': 'ESC-50-master/audio',
    'arca23k': 'ARCA23K/ARCA23K.audio/ARCA23K.audio',
    'urban8k': 'UrbanSound8K/audio/foldall',
    'dbr': 'dbr-dataset/dataset/dograin',
    'freesound': 'freesound/outside',
    'random-noise': 'random-noise',
}

In [ ]:
# common imports
import os
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# modeling imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torchvision.models as models
from safetensors.torch import save_file, load_file, safe_open
import json

# osprey package
from osprey import (
    SpectrogramDataset,
    reformat_image,
)

# Check if a GPU is available
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU instead.")

In [ ]:
weights_map = {
    'efficientnet_b0': models.EfficientNet_B0_Weights.DEFAULT,
    'efficientnet_b1': models.EfficientNet_B1_Weights.DEFAULT,
    'efficientnet_b2': models.EfficientNet_B2_Weights.DEFAULT,
    'efficientnet_b3': models.EfficientNet_B3_Weights.DEFAULT,
    'mobilenet_v3_small': models.MobileNet_V3_Small_Weights.DEFAULT,
    'mobilenet_v3_large': models.MobileNet_V3_Large_Weights.DEFAULT,
}

with safe_open(model_path, framework='pt', device='cpu') as f:
    m = f.metadata() or {}
    try:
        num_classes = int(m['num_classes'])
    except KeyError as exc:
        raise KeyError('Model metadata is missing num_classes') from exc
    try:
        model_name = m['model_name']
    except KeyError as exc:
        raise KeyError('Model metadata is missing model_name') from exc
    try:
        label_classes = json.loads(m['label_classes'])
    except KeyError as exc:
        raise KeyError('Model metadata is missing label_classes') from exc

weights = weights_map[model_name]
model_fn = getattr(models, model_name)
model = model_fn(weights=weights)

in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, num_classes)

state_dict = load_file(model_path)
model.load_state_dict(state_dict)

model.to(device)

print(f'Loaded the model successfully: {model_name}')
print(f'Number of classes: {num_classes}')
print(f'Label classes: {label_classes}')

In [ ]:
# table with metadata about audio files
table = pd.read_csv(input_path)
if n_samples > 0: # only for dev
    table = table.sample(
        n=min(n_samples, len(table)),
        random_state=42,
    ).reset_index(drop=True)

# label encoder not actually used
# but required parameter for SpectrogramDataset
le = LabelEncoder() # label encoder
le.fit(table['primary_label'])

# optional: overwrite active dataset/dataloader to use the small subset
dataset = SpectrogramDataset(
    table,
    le,
    base_folder=base_folder,
    collection_map=collection_map,
    encode_labels_onehot=False,
    mel_time_size=mel_time_size,
    label_smoothing_alpha=0.,
 )
dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers, shuffle=False)
num_batches = len(dataloader)

In [ ]:
curr_time = time.time()

model.eval()

# Initialize empty lists to hold batch tensors
y_pred_all = []

batch_number = 0
print(f'Progress chart:')
print(f'---------------')
with torch.no_grad():
    for X, Y in dataloader:
        # Move everything to device immediately
        X = X.to(device)
        Z = reformat_image(X,) 
        logits = model(Z)
        # Not even using Y
        # Inference based on loaded model
        y_pred_all.append(logits)
        batch_number += 1
        if (batch_number % 50) == 0:
            print(f'{batch_number/num_batches*100:.2f}%')


# Concatenate all batches at once and move to CPU only once
y_logits = torch.cat(y_pred_all).cpu().numpy()
y_probs = 1 / (1 + np.exp(-y_logits))  # sigmoid(logits)

print()
print(f"Inference time: {time.time() - curr_time:.4f} seconds")

In [ ]:
# Add probability columns (one per label)
for j, cname in enumerate(label_classes):
    table[cname] = y_probs[:, j].astype(float)

# save
table.to_csv(output_path, index=False)